# 🍎 CPT Newton Trainer (v1)
Entrenamiento de Interacción y Colisiones (Transferencia de Momento).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import io

# 1. Datos Incrustados
csv_data = """dx_ab,dy_ab,vx_a,vy_a,dx_bt,dy_bt,vx_b,vy_b,success
-26.64,39.87,-5.00,0.00,-221.43,-116.44,0.00,0.00,0
-20.45,36.91,-10.00,0.00,-235.52,-177.51,0.00,0.00,0
-30.01,30.97,5.00,0.00,-21.18,-73.00,0.00,0.00,0
-40.31,-25.36,-5.00,10.00,-43.04,-110.55,0.00,0.00,0
18.98,21.62,0.00,10.00,-300.26,-190.78,0.00,0.00,0
46.99,40.31,-5.00,5.00,-374.94,37.12,0.00,0.00,0
-7.16,43.08,10.00,0.00,-83.21,36.87,0.00,0.00,0
47.45,8.22,-5.00,0.00,-329.96,-233.99,0.00,0.00,0
7.04,-30.69,5.00,5.00,-82.27,26.12,0.00,0.00,0
14.57,35.52,-5.00,-5.00,-281.15,-137.39,0.00,0.00,0
-1.82,49.13,-5.00,10.00,-378.52,-10.16,0.00,0.00,0
36.42,17.33,5.00,10.00,-376.51,-3.71,0.00,0.00,0
32.97,-32.67,5.00,-10.00,-67.42,-134.24,0.00,0.00,0
-35.55,-20.35,-10.00,5.00,-175.59,27.75,0.00,0.00,0
10.23,-28.39,0.00,10.00,-161.54,21.10,0.00,0.00,0
-37.54,-30.07,-10.00,10.00,-38.36,-2.29,0.00,0.00,0
-27.64,-40.67,-10.00,10.00,-264.48,-191.66,0.00,0.00,0
-43.03,-42.67,10.00,10.00,-61.14,-1.53,0.00,0.00,0
-42.08,-39.56,-5.00,5.00,-312.29,-137.15,0.00,0.00,0
33.87,14.96,-10.00,5.00,-255.05,-85.33,0.00,0.00,0
48.16,-19.45,0.00,-5.00,-156.26,-43.18,0.00,0.00,0
47.50,-32.00,5.00,0.00,-55.56,19.28,0.00,0.00,0
-7.52,34.36,-10.00,-10.00,-97.77,-223.11,0.00,0.00,0
9.82,23.84,10.00,5.00,-212.89,-106.47,0.00,0.00,0
8.82,26.46,-5.00,-5.00,-29.87,-85.76,0.00,0.00,0
-4.74,45.30,10.00,-10.00,-226.32,-153.44,0.00,0.00,0
-31.16,7.18,10.00,10.00,-204.88,3.29,0.00,0.00,0
-46.79,8.65,5.00,0.00,-360.51,-62.72,0.00,0.00,0
12.82,28.42,0.00,-10.00,-380.28,-125.76,0.00,0.00,0
-32.59,26.89,5.00,5.00,-261.14,-71.43,0.00,0.00,0
20.37,-22.74,-5.00,10.00,-396.12,-24.52,0.00,0.00,0
38.49,19.06,-10.00,-5.00,-192.37,-233.14,0.00,0.00,0
-21.68,21.62,10.00,-5.00,-67.82,-198.60,0.00,0.00,0
-22.31,31.04,-5.00,5.00,-385.59,32.47,0.00,0.00,0
36.84,-29.82,-5.00,-10.00,-92.77,-8.19,0.00,0.00,0
-6.30,42.20,10.00,-5.00,-143.01,-129.16,0.00,0.00,0
-20.23,-17.34,0.00,-10.00,-161.53,-203.53,0.00,0.00,0
-24.51,39.83,-10.00,-10.00,-201.22,-209.69,0.00,0.00,0
-34.56,20.16,0.00,0.00,-106.78,-20.44,0.00,0.00,0
47.21,-49.16,-5.00,-10.00,-35.07,-26.53,0.00,0.00,0
37.42,-33.77,0.00,10.00,-214.90,-60.20,0.00,0.00,0
-35.63,-36.80,-10.00,0.00,-20.70,-116.44,0.00,0.00,0
-48.10,-27.06,0.00,5.00,-45.35,-59.39,0.00,0.00,0
27.95,41.97,0.00,10.00,-267.04,44.26,0.00,0.00,0
18.66,27.53,-5.00,-10.00,-196.24,-236.25,0.00,0.00,0
42.31,23.72,10.00,0.00,-107.98,36.95,0.00,0.00,0
-47.34,25.94,10.00,5.00,-398.15,4.74,0.00,0.00,0
12.67,41.11,5.00,0.00,-153.89,-155.22,0.00,0.00,0
31.30,40.93,10.00,0.00,-143.62,-174.15,0.00,0.00,0
-33.42,43.90,5.00,5.00,-388.30,-119.88,0.00,0.00,0
28.77,49.41,5.00,0.00,-257.80,-243.10,0.00,0.00,0
41.39,15.38,0.00,10.00,-145.67,28.40,0.00,0.00,0
8.68,29.84,0.00,-5.00,-166.41,-230.56,0.00,0.00,0
-36.77,-23.39,5.00,-5.00,-126.16,-165.38,0.00,0.00,0
-46.97,2.18,5.00,5.00,-342.78,-21.85,0.00,0.00,0
44.09,14.94,-5.00,-5.00,-338.46,4.28,0.00,0.00,0
-40.22,10.71,-5.00,5.00,-293.83,-103.22,0.00,0.00,0
-8.42,-33.77,0.00,-5.00,-41.55,-225.74,0.00,0.00,0
43.43,32.46,0.00,-5.00,-278.64,-245.10,0.00,0.00,0
8.88,49.11,-10.00,-5.00,-61.68,-168.78,0.00,0.00,0
-49.56,-19.20,-5.00,0.00,-259.72,6.48,0.00,0.00,0
-22.20,15.65,10.00,-5.00,-266.13,-168.33,0.00,0.00,0
18.36,20.27,-5.00,0.00,-262.60,42.36,0.00,0.00,0
-6.54,-49.48,10.00,0.00,-222.86,-205.48,0.00,0.00,0
20.97,40.21,-10.00,10.00,-285.01,-207.07,0.00,0.00,0
9.66,35.49,10.00,0.00,-253.35,-101.88,0.00,0.00,0
-41.64,-19.04,5.00,0.00,-287.92,-178.67,0.00,0.00,0
27.30,-48.19,-5.00,10.00,-187.49,30.88,0.00,0.00,0
44.16,40.99,5.00,-5.00,-278.38,-101.10,0.00,0.00,0
-30.32,-47.15,10.00,-5.00,-221.74,-223.48,0.00,0.00,0
36.13,45.23,-5.00,5.00,-324.49,-156.67,0.00,0.00,0
-23.88,-22.56,-10.00,-10.00,-21.97,-102.58,0.00,0.00,0
-37.51,36.58,5.00,-5.00,-128.91,-223.94,0.00,0.00,0
39.89,22.16,5.00,0.00,-101.76,-163.95,0.00,0.00,0
-28.03,31.39,-5.00,5.00,-163.76,18.45,0.00,0.00,0
19.95,-36.38,5.00,5.00,-210.12,-183.66,0.00,0.00,0
2.36,41.81,10.00,-5.00,-230.64,-148.12,0.00,0.00,0
-33.73,26.04,10.00,5.00,-265.22,-98.00,0.00,0.00,0
45.62,48.54,-10.00,0.00,-207.72,-94.81,0.00,0.00,0
47.77,-30.64,0.00,10.00,-206.10,-3.34,0.00,0.00,0
-17.10,-22.86,0.00,-5.00,-254.77,27.14,0.00,0.00,0
-12.13,45.44,-5.00,0.00,-382.41,-5.35,0.00,0.00,0
-26.02,-28.44,0.00,-10.00,-383.64,-221.78,0.00,0.00,0
-10.42,25.48,10.00,-10.00,-328.16,-232.82,0.00,0.00,0
-38.35,-23.78,-10.00,10.00,-208.02,-86.07,0.00,0.00,0
13.90,42.43,-5.00,5.00,-26.27,-180.94,0.00,0.00,0
25.54,-20.79,-10.00,-5.00,-304.09,-83.55,0.00,0.00,0
-41.34,45.77,5.00,5.00,-35.16,-115.57,0.00,0.00,0
23.94,-24.65,0.00,-5.00,-316.93,-48.71,0.00,0.00,0
-23.91,42.73,-10.00,10.00,-32.81,-90.76,0.00,0.00,0
28.96,8.35,10.00,-10.00,-145.04,-85.40,0.00,0.00,0
39.46,-14.85,-10.00,5.00,-391.96,-222.37,0.00,0.00,0
17.38,-47.12,-10.00,-5.00,-267.40,-202.50,0.00,0.00,0
-11.82,-30.09,5.00,5.00,-337.41,-227.27,0.00,0.00,0
-40.99,19.59,5.00,-10.00,-379.17,3.85,0.00,0.00,0
46.35,26.29,-10.00,10.00,-1.56,-112.01,0.00,0.00,0
41.21,44.71,10.00,10.00,-208.79,-217.77,0.00,0.00,0
38.55,45.82,-10.00,10.00,-62.46,-138.55,0.00,0.00,0
-28.71,-21.14,0.00,-10.00,-291.89,-97.68,0.00,0.00,0
37.30,-43.74,5.00,-5.00,-167.03,-190.87,0.00,0.00,0
-16.52,42.03,10.00,-10.00,-336.59,-181.07,0.00,0.00,0
20.59,-31.17,-10.00,-5.00,-31.23,-91.48,0.00,0.00,0
-41.30,-8.54,-5.00,-10.00,-212.86,-178.28,0.00,0.00,0
37.12,32.91,-5.00,-5.00,-355.23,-245.09,0.00,0.00,0
31.82,25.10,10.00,0.00,-346.08,-137.08,0.00,0.00,0
47.52,-9.78,5.00,0.00,-275.87,-149.48,0.00,0.00,0
47.57,-16.20,5.00,10.00,-172.77,47.58,0.00,0.00,0
40.79,33.07,10.00,5.00,-227.08,-51.44,0.00,0.00,0
36.91,-4.74,0.00,5.00,-397.25,-124.32,0.00,0.00,0
34.43,-45.75,10.00,0.00,-228.68,-186.11,0.00,0.00,0
41.14,-6.38,-5.00,-10.00,-311.04,-125.36,0.00,0.00,0
-26.66,13.32,10.00,-5.00,-169.21,4.20,0.00,0.00,0
-47.09,13.22,5.00,10.00,-217.17,4.00,0.00,0.00,0
33.08,44.39,-5.00,-10.00,-295.26,-111.15,0.00,0.00,0
33.80,-27.78,10.00,0.00,-390.57,-234.39,0.00,0.00,0
-24.35,46.47,-10.00,0.00,-373.04,-207.00,0.00,0.00,0
34.69,-29.37,-10.00,-5.00,-23.90,12.29,0.00,0.00,0
18.59,24.91,-5.00,0.00,-167.45,-83.73,0.00,0.00,0
-5.92,31.48,0.00,5.00,-387.94,-148.34,0.00,0.00,0
30.95,-21.79,10.00,-10.00,-319.12,-142.48,0.00,0.00,0
24.85,48.99,5.00,5.00,-63.42,-168.00,0.00,0.00,0
-23.54,-49.89,10.00,-5.00,-60.21,-16.44,0.00,0.00,0
47.90,9.16,5.00,10.00,-26.99,-108.22,0.00,0.00,0
-29.50,-4.60,-10.00,-5.00,-350.97,-199.27,0.00,0.00,0
-41.38,11.02,-5.00,-10.00,-244.59,-112.26,0.00,0.00,0
18.08,20.32,-10.00,-5.00,-116.26,-40.79,0.00,0.00,0
-33.16,-25.87,0.00,0.00,-230.55,-68.81,0.00,0.00,0
-8.66,38.09,5.00,5.00,-122.79,-39.81,0.00,0.00,0
43.95,-49.41,5.00,10.00,-393.32,-87.33,0.00,0.00,0
-12.51,-43.00,-5.00,-5.00,-8.26,-209.61,0.00,0.00,0
-49.55,26.50,-10.00,-5.00,-31.82,-205.60,0.00,0.00,0
-10.31,-23.86,5.00,10.00,-238.92,-70.21,0.00,0.00,0
27.37,33.21,10.00,-10.00,-247.20,44.69,0.00,0.00,0
-25.81,3.84,-10.00,5.00,-127.04,-4.32,0.00,0.00,0
-32.42,-41.84,10.00,-10.00,-151.76,-109.12,0.00,0.00,0
-31.50,-45.92,-10.00,-5.00,-31.20,-241.58,0.00,0.00,0
16.95,29.73,5.00,10.00,-163.10,-165.01,0.00,0.00,0
-24.93,32.58,5.00,10.00,-332.15,-118.03,0.00,0.00,0
17.96,22.42,0.00,0.00,-217.30,-95.14,0.00,0.00,0
31.26,28.18,10.00,-10.00,-313.87,-193.67,0.00,0.00,0
-6.27,29.20,0.00,-10.00,-301.98,23.72,0.00,0.00,0
-34.31,18.76,0.00,-5.00,-50.62,-151.84,0.00,0.00,0
3.95,-37.56,10.00,10.00,-273.24,-165.99,0.00,0.00,0
-17.96,-47.23,-5.00,-10.00,-86.61,-96.24,0.00,0.00,0
31.15,1.06,5.00,0.00,-236.40,48.38,0.00,0.00,0
-21.99,-30.92,5.00,5.00,-197.70,-226.63,0.00,0.00,0
4.61,-40.98,-5.00,-5.00,-317.27,-165.01,0.00,0.00,0
31.47,-8.09,-5.00,10.00,-384.60,-101.56,0.00,0.00,0
39.34,22.71,0.00,-10.00,-28.49,-191.61,0.00,0.00,0
14.47,28.97,5.00,5.00,-262.59,29.41,0.00,0.00,0
31.35,17.66,-5.00,-5.00,-7.26,-226.60,0.00,0.00,0
-16.71,-34.35,-10.00,-5.00,-313.97,-98.33,0.00,0.00,0
-48.36,8.71,-5.00,10.00,-80.95,35.65,0.00,0.00,0
-26.79,6.72,5.00,10.00,-221.70,-249.31,0.00,0.00,0
-41.50,-38.76,-5.00,-10.00,-338.27,-156.23,0.00,0.00,0
-37.52,-27.44,-5.00,10.00,-19.42,-16.87,0.00,0.00,0
-8.36,-45.01,-10.00,-10.00,-244.91,-75.49,0.00,0.00,0
-24.81,43.69,0.00,-10.00,-329.16,-243.44,0.00,0.00,0
-23.54,-23.91,5.00,0.00,-351.56,-244.14,0.00,0.00,0
-29.40,2.06,10.00,-10.00,-326.45,-141.87,0.00,0.00,0
29.01,-15.60,0.00,-5.00,-236.61,-175.74,0.00,0.00,0
-47.85,-21.42,10.00,0.00,-188.08,-32.22,0.00,0.00,0
29.78,-22.85,-5.00,0.00,-246.32,-188.56,0.00,0.00,0
35.84,4.04,0.00,10.00,-23.58,-30.66,0.00,0.00,0
3.01,-27.72,5.00,5.00,-155.71,-230.52,0.00,0.00,0
-46.25,34.83,-5.00,10.00,-211.91,-68.06,0.00,0.00,0
-49.84,26.12,0.00,-5.00,-277.81,-135.71,0.00,0.00,0
-46.08,6.31,-10.00,0.00,-110.46,-193.52,0.00,0.00,0
49.97,-44.28,-10.00,5.00,-148.91,36.18,0.00,0.00,0
-45.80,-32.95,-5.00,0.00,-269.41,-149.05,0.00,0.00,0
36.47,-13.91,0.00,0.00,-376.54,-233.95,0.00,0.00,0
36.53,-8.18,-10.00,0.00,-374.83,-214.56,0.00,0.00,0
-32.04,16.65,5.00,0.00,-144.78,-177.43,0.00,0.00,0
-21.68,-17.98,-10.00,0.00,-142.20,-85.15,0.00,0.00,0
-48.55,-18.44,10.00,10.00,-120.18,-163.00,0.00,0.00,0
32.79,27.83,5.00,5.00,-203.86,-189.51,0.00,0.00,0
4.07,43.02,-10.00,0.00,-140.44,-230.24,0.00,0.00,0
-38.41,19.88,-5.00,10.00,-9.10,-129.57,0.00,0.00,0
33.56,-44.00,10.00,10.00,-26.94,-228.84,0.00,0.00,0
-22.61,18.48,-5.00,-5.00,-11.48,-117.21,0.00,0.00,0
48.83,45.29,0.00,5.00,-222.82,-9.21,0.00,0.00,0
-5.07,25.46,-5.00,10.00,-52.07,-165.12,0.00,0.00,0
-20.76,-24.33,-5.00,10.00,-16.16,-135.47,0.00,0.00,0
11.58,40.69,0.00,5.00,-373.40,21.92,0.00,0.00,0
-20.73,33.99,-5.00,5.00,-93.33,-231.86,0.00,0.00,0
5.86,43.17,5.00,-10.00,-121.91,-198.77,0.00,0.00,0
-8.15,33.70,10.00,0.00,-91.65,-141.39,0.00,0.00,0
35.58,41.07,-5.00,5.00,-222.70,-149.06,0.00,0.00,0
24.67,-39.16,10.00,5.00,-351.29,-207.30,0.00,0.00,0
-23.35,-44.26,10.00,-10.00,-307.04,-6.41,0.00,0.00,0
14.87,23.29,0.00,-10.00,-8.51,-197.68,0.00,0.00,0
29.13,48.84,10.00,0.00,-361.13,-1.16,0.00,0.00,0
44.27,-45.48,-10.00,-10.00,-65.55,-213.94,0.00,0.00,0
-3.81,47.36,-10.00,5.00,-274.86,-34.43,0.00,0.00,0
-43.95,-26.24,-10.00,-5.00,-372.00,-248.16,0.00,0.00,0
-7.44,25.95,10.00,10.00,-106.36,-158.93,0.00,0.00,0
-35.37,-49.83,10.00,-5.00,-192.37,-145.22,0.00,0.00,0
37.87,29.73,5.00,5.00,-47.88,-91.95,0.00,0.00,0
-49.85,-24.00,5.00,0.00,-170.55,-5.02,0.00,0.00,0
-15.67,-25.09,-10.00,-10.00,-348.35,22.13,0.00,0.00,0
13.16,-28.94,-5.00,0.00,-23.23,-63.61,0.00,0.00,0
-2.23,29.47,0.00,-5.00,-353.34,-49.49,0.00,0.00,0
-47.45,5.93,-5.00,5.00,-209.80,-127.64,0.00,0.00,0
24.74,-49.18,-10.00,-10.00,-289.25,-122.78,0.00,0.00,0
49.92,2.16,5.00,10.00,-17.94,-140.20,0.00,0.00,0
-48.32,10.37,5.00,0.00,-395.99,-132.08,0.00,0.00,0
48.62,-23.21,10.00,10.00,-373.55,-127.84,0.00,0.00,0
-9.47,-35.18,-10.00,10.00,-242.03,-14.72,0.00,0.00,0
-42.04,-15.27,0.00,5.00,-260.02,-133.78,0.00,0.00,0
7.12,-27.83,0.00,0.00,-18.26,-111.34,0.00,0.00,0
-13.20,-39.92,0.00,5.00,-215.06,49.76,0.00,0.00,0
26.65,-46.62,-5.00,-10.00,-249.40,-190.84,0.00,0.00,0
4.50,-37.83,0.00,-5.00,-351.04,-192.41,0.00,0.00,0
4.80,27.83,0.00,0.00,-330.42,-56.65,0.00,0.00,0
-23.12,-41.42,10.00,-10.00,-43.67,26.90,0.00,0.00,0
-42.70,28.61,-10.00,5.00,-183.20,-10.97,0.00,0.00,0
-14.32,-49.10,-10.00,0.00,-340.81,-19.65,0.00,0.00,0
-30.82,-0.08,0.00,-10.00,-152.66,9.65,0.00,0.00,0
-49.18,23.44,-10.00,-5.00,-253.17,-239.98,0.00,0.00,0
-33.57,48.13,5.00,-5.00,-163.25,47.93,0.00,0.00,0
16.60,-38.20,10.00,5.00,-88.89,-205.14,0.00,0.00,0
42.66,-12.76,-10.00,-10.00,-6.08,-225.65,0.00,0.00,0
-24.30,5.89,-5.00,0.00,-91.98,-132.90,0.00,0.00,0
-27.39,33.79,5.00,-10.00,-378.76,30.19,0.00,0.00,0
-23.55,-42.23,0.00,-5.00,-208.08,-166.05,0.00,0.00,0
11.17,-41.04,0.00,-10.00,-87.10,-235.75,0.00,0.00,0
-1.68,-49.33,0.00,5.00,-341.39,-182.99,0.00,0.00,0
15.39,38.22,-5.00,-5.00,-159.39,37.84,0.00,0.00,0
-45.34,-19.45,0.00,0.00,-68.20,-3.91,0.00,0.00,0
-49.19,-48.14,-5.00,10.00,-316.34,49.86,0.00,0.00,0
-40.65,-46.44,0.00,0.00,-182.06,-2.77,0.00,0.00,0
-48.54,-49.21,-5.00,0.00,-333.77,36.49,0.00,0.00,0
38.15,-10.61,5.00,0.00,-77.24,26.38,0.00,0.00,0
16.28,-42.45,-10.00,5.00,-273.53,17.87,0.00,0.00,0
13.93,-29.78,5.00,-5.00,-351.42,-117.56,0.00,0.00,0
35.02,47.31,0.00,5.00,-0.66,-30.26,0.00,0.00,0
-35.83,-24.33,5.00,0.00,-103.52,-78.16,0.00,0.00,0
18.88,-45.58,-5.00,5.00,-179.33,-206.85,0.00,0.00,0
47.18,-41.65,5.00,-10.00,-211.13,-91.28,0.00,0.00,0
31.13,-12.39,-10.00,-10.00,-173.82,-223.47,0.00,0.00,0
-41.18,-48.58,-5.00,10.00,-84.81,-66.21,0.00,0.00,0
29.70,-36.52,-10.00,-10.00,-299.32,-231.76,0.00,0.00,0
-32.39,-29.22,-5.00,10.00,-50.30,-145.93,0.00,0.00,0
-46.02,-42.74,5.00,-5.00,-236.10,-27.21,0.00,0.00,0
5.67,-26.27,5.00,10.00,-32.27,-4.97,0.00,0.00,0
-19.74,-26.26,-5.00,-10.00,-386.02,-179.05,0.00,0.00,0
36.91,-26.50,-5.00,-5.00,-365.78,-207.73,0.00,0.00,0
-34.49,-3.60,10.00,10.00,-123.53,-130.64,0.00,0.00,0
-25.46,-7.67,-5.00,5.00,-45.88,-208.52,0.00,0.00,0
33.90,35.82,-10.00,10.00,-391.15,-89.02,0.00,0.00,0
-42.41,-14.11,10.00,-10.00,-150.17,-222.49,0.00,0.00,0
-47.74,-26.60,10.00,-10.00,-180.60,49.44,0.00,0.00,0
-40.12,25.31,5.00,-5.00,-122.90,-5.58,0.00,0.00,0
-38.69,38.61,0.00,10.00,-304.19,-179.60,0.00,0.00,0
45.84,40.06,10.00,5.00,-354.61,26.66,0.00,0.00,0
2.62,25.30,-10.00,-5.00,-349.79,-54.74,0.00,0.00,0
22.50,12.31,-5.00,0.00,-109.11,-86.84,0.00,0.00,0
-9.05,43.18,10.00,-5.00,-15.07,16.73,0.00,0.00,0
-42.25,-15.94,-10.00,10.00,-273.91,-132.79,0.00,0.00,0
41.33,-37.67,10.00,0.00,-100.54,-73.16,0.00,0.00,0
-8.72,27.10,-5.00,10.00,-130.58,-102.55,0.00,0.00,0
-35.20,35.78,0.00,0.00,-99.82,-50.93,0.00,0.00,0
-33.86,33.48,-5.00,0.00,-16.50,-13.08,0.00,0.00,0
7.59,32.17,-5.00,-10.00,-96.27,10.61,0.00,0.00,0
-46.21,23.49,-5.00,-10.00,-311.73,-123.08,0.00,0.00,0
-48.23,45.39,-10.00,-5.00,-393.88,43.63,0.00,0.00,0
47.78,-34.94,-10.00,10.00,-356.37,34.80,0.00,0.00,0
-40.38,-2.18,5.00,-10.00,-393.17,-13.87,0.00,0.00,0
34.70,-31.07,-10.00,-10.00,-279.24,-152.99,0.00,0.00,0
-32.67,4.96,0.00,5.00,-186.78,-194.57,0.00,0.00,0
46.48,47.53,0.00,5.00,-321.86,-140.99,0.00,0.00,0
-12.83,-40.68,5.00,10.00,-298.14,5.37,0.00,0.00,0
39.72,42.24,-5.00,10.00,-118.91,-104.32,0.00,0.00,0
17.55,-33.96,5.00,0.00,-48.95,-208.56,0.00,0.00,0
-47.45,16.05,-10.00,-5.00,-378.90,-154.24,0.00,0.00,0
41.98,-19.43,-5.00,-10.00,-275.55,-238.46,0.00,0.00,0
25.84,8.26,-5.00,10.00,-243.59,-13.69,0.00,0.00,0
29.78,-24.89,5.00,10.00,-293.86,-215.14,0.00,0.00,0
37.02,-16.51,-10.00,10.00,-255.85,-172.58,0.00,0.00,0
-39.98,2.69,-10.00,10.00,-372.41,-1.28,0.00,0.00,0
10.73,49.97,-10.00,-5.00,-259.78,-58.81,0.00,0.00,0
-47.88,47.93,0.00,10.00,-81.53,-148.24,0.00,0.00,0
29.90,47.31,-10.00,-10.00,-340.26,-59.50,0.00,0.00,0
37.26,30.25,5.00,-5.00,-118.02,8.24,0.00,0.00,0
43.68,16.74,0.00,-10.00,-73.63,-64.88,0.00,0.00,0
-33.66,23.34,0.00,10.00,-134.43,-192.96,0.00,0.00,0
34.57,-24.29,-5.00,5.00,-304.96,-85.48,0.00,0.00,0
-31.62,-22.22,5.00,-5.00,-222.09,-102.00,0.00,0.00,0
-47.29,22.16,-10.00,10.00,-316.33,-136.71,0.00,0.00,0
-35.75,28.81,0.00,5.00,-284.96,-109.47,0.00,0.00,0
-27.11,-34.63,5.00,0.00,-80.82,47.32,0.00,0.00,0
-17.58,-41.59,5.00,10.00,-271.95,-65.95,0.00,0.00,0
-34.13,31.23,10.00,10.00,-53.61,-37.83,0.00,0.00,0
-31.87,27.25,-5.00,-5.00,-3.41,-35.15,0.00,0.00,0
-12.18,31.77,-5.00,0.00,-257.15,32.75,0.00,0.00,0
-47.06,23.80,-5.00,5.00,-250.63,5.30,0.00,0.00,0
-32.49,-28.29,-5.00,-5.00,-204.14,-211.83,0.00,0.00,0
-26.94,28.00,-5.00,-5.00,-78.45,49.37,0.00,0.00,0
21.60,29.52,10.00,-5.00,-54.83,-124.63,0.00,0.00,0
-47.75,30.07,5.00,5.00,-87.04,-31.93,0.00,0.00,0
-49.43,-13.55,-10.00,-5.00,-84.94,-137.61,0.00,0.00,0
-41.73,42.80,5.00,-10.00,-62.90,-93.99,0.00,0.00,0
-15.72,29.00,-10.00,-5.00,-160.80,-10.91,0.00,0.00,0
25.86,3.72,5.00,-5.00,-199.11,-115.08,0.00,0.00,0
-34.70,42.81,-5.00,-10.00,-248.15,-13.76,0.00,0.00,0
-16.63,45.47,10.00,-5.00,-47.64,-176.16,0.00,0.00,0
27.83,2.15,10.00,-10.00,-389.41,-166.61,0.00,0.00,0
31.12,19.71,0.00,0.00,-362.02,-108.41,0.00,0.00,0
35.55,-23.57,5.00,-10.00,-100.15,-45.34,0.00,0.00,0
36.54,7.60,-5.00,-10.00,-311.60,25.83,0.00,0.00,0
33.84,-11.44,-10.00,0.00,-196.89,35.93,0.00,0.00,0
46.73,15.12,0.00,-10.00,-176.26,-88.50,0.00,0.00,0
26.86,-8.56,-10.00,5.00,-172.99,4.12,0.00,0.00,0
-23.37,49.08,0.00,0.00,-181.60,-68.98,0.00,0.00,0
36.85,-3.69,-5.00,0.00,-151.74,-30.82,0.00,0.00,0
-41.07,-46.98,-5.00,-5.00,-368.83,-178.20,0.00,0.00,0
-36.59,20.75,-10.00,0.00,-22.83,-117.64,0.00,0.00,0
-11.51,30.12,-5.00,-5.00,-336.00,-10.67,0.00,0.00,0
37.94,1.97,-10.00,0.00,-349.39,-46.21,0.00,0.00,0
27.03,-39.75,-10.00,5.00,-139.52,-48.65,0.00,0.00,0
-28.56,24.34,-5.00,-5.00,-42.47,-149.65,0.00,0.00,0
1.86,-48.27,10.00,10.00,-90.98,-203.56,0.00,0.00,0
29.81,41.78,0.00,5.00,-338.00,-128.81,0.00,0.00,0
-46.50,1.26,-10.00,-10.00,-88.26,-86.57,0.00,0.00,0
40.08,22.17,-5.00,-10.00,-8.02,34.78,0.00,0.00,0
-37.62,-1.73,-5.00,-5.00,-245.47,-6.82,0.00,0.00,0
44.03,3.21,-10.00,-5.00,-239.01,-121.02,0.00,0.00,0
48.55,27.96,-5.00,5.00,-100.91,-32.91,0.00,0.00,0
-39.18,43.88,-10.00,-10.00,-157.32,-106.27,0.00,0.00,0
4.17,34.77,5.00,-10.00,-352.59,44.59,0.00,0.00,0
11.63,31.24,-10.00,5.00,-36.46,-187.01,0.00,0.00,0
44.36,7.44,-10.00,-10.00,-333.83,-100.11,0.00,0.00,0
-43.15,1.23,10.00,0.00,-212.66,-90.30,0.00,0.00,0
21.38,-49.46,-5.00,10.00,-322.38,-104.37,0.00,0.00,0
29.24,46.32,5.00,5.00,-189.02,-13.32,0.00,0.00,0
45.90,1.84,10.00,10.00,-152.07,-183.56,0.00,0.00,0
37.66,45.13,0.00,10.00,-329.75,-147.92,0.00,0.00,0
8.43,-47.72,-5.00,5.00,-371.40,-218.96,0.00,0.00,0
6.72,39.73,5.00,-5.00,-31.11,-190.50,0.00,0.00,0
-45.39,-43.56,5.00,-10.00,-328.02,-192.08,0.00,0.00,0
-5.24,48.96,5.00,-5.00,-146.37,-236.20,0.00,0.00,0
-41.84,-23.45,0.00,0.00,-251.13,-115.24,0.00,0.00,0
4.85,28.49,10.00,-10.00,-157.85,-109.06,0.00,0.00,0
28.48,12.75,5.00,0.00,-350.46,-145.55,0.00,0.00,0
-38.74,40.26,-10.00,-5.00,-179.02,-205.37,0.00,0.00,0
-0.08,-34.70,0.00,5.00,-127.19,-118.89,0.00,0.00,0
33.53,-20.81,5.00,5.00,-10.47,-105.67,0.00,0.00,0
42.80,2.81,-10.00,5.00,-81.38,-90.25,0.00,0.00,0
19.17,41.73,-10.00,0.00,-359.56,-241.33,0.00,0.00,0
-2.82,48.16,5.00,-10.00,-363.71,-51.05,0.00,0.00,0
-23.88,35.71,5.00,-5.00,-25.57,-35.32,0.00,0.00,0
31.84,11.48,5.00,-10.00,-256.43,-184.23,0.00,0.00,0
-27.41,-33.14,-5.00,-5.00,-397.07,-13.23,0.00,0.00,0
33.07,-45.37,-5.00,-10.00,-338.55,-35.13,0.00,0.00,0
37.39,4.50,5.00,-5.00,-162.63,0.34,0.00,0.00,0
18.99,46.70,10.00,-10.00,-340.26,-221.43,0.00,0.00,0
45.72,28.73,10.00,-5.00,-380.10,-93.79,0.00,0.00,0
15.43,36.13,-10.00,0.00,-262.86,35.78,0.00,0.00,0
25.18,15.03,-5.00,0.00,-318.01,-95.37,0.00,0.00,0
8.71,-48.32,10.00,-5.00,-392.22,-150.07,0.00,0.00,0
-35.28,-48.00,-10.00,5.00,-15.41,-228.16,0.00,0.00,0
19.39,36.60,0.00,5.00,-388.79,-205.82,0.00,0.00,0
39.65,-0.83,0.00,-5.00,-109.50,-36.81,0.00,0.00,0
23.84,12.26,0.00,5.00,-246.94,-94.28,0.00,0.00,0
8.59,-40.13,-5.00,-10.00,-150.32,-52.66,0.00,0.00,0
-28.09,44.57,-10.00,5.00,-59.20,-171.88,0.00,0.00,0
-45.39,28.04,-5.00,-5.00,-21.14,-89.67,0.00,0.00,0
2.04,-39.20,-5.00,0.00,-33.13,-24.62,0.00,0.00,0
26.17,40.78,10.00,5.00,-139.55,-104.09,0.00,0.00,0
-41.71,-36.02,0.00,-5.00,-73.63,-99.50,0.00,0.00,0
-28.21,8.30,5.00,5.00,-29.06,-94.01,0.00,0.00,0
-47.58,41.11,-5.00,-10.00,-316.15,-15.99,0.00,0.00,0
26.53,-9.80,-10.00,-5.00,-399.38,-80.05,0.00,0.00,0
26.54,-20.21,0.00,0.00,-330.41,-235.08,0.00,0.00,0
-47.03,-9.09,0.00,0.00,-193.52,-95.38,0.00,0.00,0
26.54,-31.46,5.00,-5.00,-71.10,-5.17,0.00,0.00,0
-14.75,34.58,0.00,0.00,-28.07,-88.20,0.00,0.00,0
31.39,37.34,10.00,-5.00,-17.73,-116.37,0.00,0.00,0
-9.56,-24.85,-5.00,10.00,-29.78,-231.38,0.00,0.00,0
20.43,-16.24,-5.00,-10.00,-303.69,-182.66,0.00,0.00,0
13.92,-33.90,5.00,-5.00,-131.91,-19.42,0.00,0.00,0
9.97,42.02,-10.00,10.00,-232.04,-241.03,0.00,0.00,0
-1.53,-29.62,0.00,10.00,-160.73,-147.18,0.00,0.00,0
12.07,-35.92,5.00,-5.00,-166.11,-19.40,0.00,0.00,0
-48.54,8.35,-10.00,5.00,-259.96,-2.73,0.00,0.00,0
23.17,-49.49,10.00,10.00,-137.49,-21.78,0.00,0.00,0
25.08,31.71,5.00,-5.00,-8.31,36.54,0.00,0.00,0
29.03,43.05,5.00,-10.00,-241.15,-27.43,0.00,0.00,0
14.28,-33.10,0.00,10.00,-275.95,-138.55,0.00,0.00,0
11.52,33.79,10.00,5.00,-252.11,-195.95,0.00,0.00,0
-18.14,-45.57,10.00,10.00,-104.21,-171.89,0.00,0.00,0
-38.48,-24.57,-10.00,-5.00,-24.53,-35.87,0.00,0.00,0
-24.80,-5.59,0.00,-10.00,-218.15,-124.33,0.00,0.00,0
-45.09,7.99,0.00,-5.00,-207.13,-38.12,0.00,0.00,0
-48.65,22.75,-10.00,-5.00,-92.49,-186.48,0.00,0.00,0
16.89,-27.56,0.00,10.00,-391.31,-79.13,0.00,0.00,0
21.54,-17.38,-5.00,-5.00,-95.81,-88.57,0.00,0.00,0
-32.86,-38.76,0.00,-5.00,-342.92,-170.00,0.00,0.00,0
28.14,19.01,-5.00,5.00,-205.64,-74.18,0.00,0.00,0
-41.30,-11.09,-5.00,10.00,-294.54,-174.30,0.00,0.00,0
-28.79,16.03,0.00,0.00,-108.35,-158.50,0.00,0.00,0
-46.74,-26.76,0.00,-10.00,-158.31,-125.73,0.00,0.00,0
19.74,38.95,10.00,-10.00,-376.87,-141.87,0.00,0.00,0
39.10,15.98,-10.00,5.00,-2.01,-97.42,0.00,0.00,0
-23.40,-29.23,-5.00,10.00,-77.32,-137.78,0.00,0.00,0
45.99,12.88,10.00,5.00,-207.75,-40.20,0.00,0.00,0
35.59,-11.85,5.00,-5.00,-245.99,-217.25,0.00,0.00,0
41.49,44.43,10.00,5.00,-22.30,-174.15,0.00,0.00,0
20.54,41.76,5.00,5.00,-35.03,-74.72,0.00,0.00,0
29.63,-48.63,-10.00,-10.00,-113.46,-72.53,0.00,0.00,0
-35.29,35.61,-10.00,-10.00,-348.26,-26.17,0.00,0.00,0
-47.21,-33.40,0.00,10.00,-56.11,-18.36,0.00,0.00,0
7.29,38.18,5.00,-5.00,-188.71,-46.93,0.00,0.00,0
-27.44,-15.54,0.00,10.00,-176.11,-162.22,0.00,0.00,0
36.64,31.38,5.00,-5.00,-299.92,-203.12,0.00,0.00,0
43.08,7.87,0.00,-10.00,-287.72,-49.65,0.00,0.00,0
-33.89,5.82,-5.00,-10.00,-255.79,-160.26,0.00,0.00,0
-2.16,-41.16,-10.00,5.00,-316.67,-126.51,0.00,0.00,0
42.12,-10.34,10.00,0.00,-189.92,-167.22,0.00,0.00,0
12.19,-49.89,10.00,5.00,-99.97,-214.32,0.00,0.00,0
32.94,16.75,10.00,0.00,-250.34,9.93,0.00,0.00,0
26.15,-20.21,-10.00,5.00,-330.60,-124.71,0.00,0.00,0
-28.89,5.31,-10.00,-10.00,-290.36,-45.56,0.00,0.00,0
37.63,-40.11,5.00,10.00,-235.19,-148.14,0.00,0.00,0
19.65,16.08,-5.00,-10.00,-188.22,-99.24,0.00,0.00,0
33.24,-36.46,-5.00,-5.00,-366.63,-240.68,0.00,0.00,0
17.68,-27.79,-10.00,0.00,-257.51,-7.55,0.00,0.00,0
-33.48,16.10,10.00,10.00,-238.11,-175.07,0.00,0.00,0
-39.67,26.75,-5.00,-10.00,-34.46,-178.99,0.00,0.00,0
-11.57,-33.02,10.00,10.00,-65.37,-148.93,0.00,0.00,0
-21.74,-46.61,-5.00,10.00,-169.87,-84.73,0.00,0.00,0
35.00,-4.22,-5.00,-10.00,-313.25,-165.32,0.00,0.00,0
14.19,-29.38,-10.00,-5.00,-187.48,-24.40,0.00,0.00,0
-5.43,-30.77,-5.00,0.00,-245.97,-198.46,0.00,0.00,0
-22.42,-13.85,-5.00,-5.00,-102.28,-72.42,0.00,0.00,0
-5.67,34.71,10.00,10.00,-72.25,12.85,0.00,0.00,0
-24.69,39.76,5.00,0.00,-398.13,9.89,0.00,0.00,0
-46.10,-6.54,10.00,10.00,-53.20,-190.19,0.00,0.00,0
35.32,30.49,-5.00,0.00,-69.90,-97.41,0.00,0.00,0
-18.12,34.51,-10.00,-5.00,-352.15,-11.48,0.00,0.00,0
-48.69,45.10,0.00,10.00,-317.59,-179.90,0.00,0.00,0
25.36,49.79,-10.00,-5.00,-287.90,-146.21,0.00,0.00,0
47.40,13.36,-10.00,-10.00,-177.61,-105.97,0.00,0.00,0
5.76,-45.48,-5.00,-10.00,-161.99,-52.71,0.00,0.00,0
46.87,-29.23,10.00,5.00,-302.07,-75.15,0.00,0.00,0
43.36,-38.90,5.00,-10.00,-171.72,-35.17,0.00,0.00,0
-17.31,22.75,-10.00,10.00,-357.14,49.10,0.00,0.00,0
31.91,3.31,-5.00,5.00,-398.25,-32.00,0.00,0.00,0
35.63,3.13,10.00,-5.00,-397.40,-192.77,0.00,0.00,0
-43.57,38.69,10.00,10.00,-51.71,-234.76,0.00,0.00,0
-41.14,14.14,5.00,0.00,-22.56,-147.57,0.00,0.00,0
-26.68,16.58,-5.00,10.00,-57.11,-9.06,0.00,0.00,0
14.35,31.26,-5.00,-10.00,-258.55,-24.41,0.00,0.00,0
34.03,38.58,5.00,0.00,-331.46,-104.49,0.00,0.00,0
11.95,29.73,-10.00,10.00,-58.52,-6.15,0.00,0.00,0
-7.84,-38.11,0.00,-5.00,-348.01,32.63,0.00,0.00,0
-18.55,-44.13,-10.00,-10.00,-304.38,28.69,0.00,0.00,0
-32.32,-8.71,-10.00,10.00,-40.97,-232.58,0.00,0.00,0
-4.32,43.45,-10.00,10.00,-137.83,2.63,0.00,0.00,0
36.72,-24.70,-5.00,0.00,-190.87,-190.13,0.00,0.00,0
-40.91,-39.76,0.00,5.00,-164.52,-95.21,0.00,0.00,0
46.54,-33.21,10.00,10.00,-319.69,-41.10,0.00,0.00,0
-15.36,-41.07,-5.00,0.00,-147.15,1.94,0.00,0.00,0
-21.11,17.41,-5.00,10.00,-114.98,-197.66,0.00,0.00,0
-49.95,5.30,-10.00,10.00,-305.08,-247.74,0.00,0.00,0
14.57,30.57,0.00,10.00,-156.88,-46.39,0.00,0.00,0
19.02,25.90,-10.00,-10.00,-202.82,-223.98,0.00,0.00,0
-24.00,35.27,0.00,-10.00,-211.72,-97.67,0.00,0.00,0
-31.82,-43.26,-5.00,-5.00,-42.91,22.61,0.00,0.00,0
47.70,-26.61,0.00,5.00,-310.75,24.97,0.00,0.00,0
21.25,-14.17,10.00,-5.00,-338.36,-162.92,0.00,0.00,0
-11.09,-43.04,-10.00,5.00,-196.60,-90.56,0.00,0.00,0
-5.38,39.96,0.00,10.00,-177.79,-96.83,0.00,0.00,0
-26.96,-48.09,10.00,-5.00,-346.92,-59.04,0.00,0.00,0
41.95,-36.29,-5.00,-10.00,-302.00,-131.16,0.00,0.00,0
-33.70,24.53,5.00,0.00,-43.97,-137.44,0.00,0.00,0
-38.08,-49.20,10.00,5.00,-100.21,-150.23,0.00,0.00,0
-29.17,26.22,5.00,5.00,-385.85,2.91,0.00,0.00,0
35.75,38.71,5.00,-5.00,-73.80,-80.40,0.00,0.00,0
19.76,22.57,-10.00,5.00,-363.65,-111.09,0.00,0.00,0
-22.54,22.44,-5.00,5.00,-37.26,-11.34,0.00,0.00,0
36.40,40.95,-10.00,10.00,-109.99,-210.44,0.00,0.00,0
-49.52,27.40,0.00,5.00,-380.20,-187.24,0.00,0.00,0
-32.15,10.12,10.00,-5.00,-384.98,-185.51,0.00,0.00,0
36.01,-31.64,5.00,-10.00,-338.97,-125.63,0.00,0.00,0
-31.93,-20.04,5.00,10.00,-10.96,-160.36,0.00,0.00,0
-23.24,14.36,5.00,0.00,-244.92,-74.74,0.00,0.00,0
-7.25,28.69,-10.00,-5.00,-293.92,-86.69,0.00,0.00,0
-41.26,28.37,10.00,-5.00,-264.70,-184.56,0.00,0.00,0
-9.84,-44.40,10.00,10.00,-113.47,-229.90,0.00,0.00,0
3.90,40.58,5.00,-5.00,-8.71,-184.58,0.00,0.00,0
40.98,-40.68,0.00,-5.00,-375.98,-177.96,0.00,0.00,0
-32.00,35.67,-5.00,-5.00,-330.40,-65.40,0.00,0.00,0
27.40,-26.57,-10.00,-10.00,-239.51,21.32,0.00,0.00,0
-14.71,-22.81,0.00,-10.00,-353.20,-163.57,0.00,0.00,0
-48.84,-45.37,5.00,-10.00,-159.41,-37.55,0.00,0.00,0
9.75,-26.72,-5.00,5.00,-195.77,-48.39,0.00,0.00,0
-35.63,27.78,-5.00,-10.00,-218.32,49.78,0.00,0.00,0
21.89,25.29,-10.00,0.00,-201.12,-121.63,0.00,0.00,0
29.24,3.22,0.00,5.00,-231.64,-47.40,0.00,0.00,0
-35.29,-32.19,-5.00,-10.00,-268.43,-153.17,0.00,0.00,0
-28.21,-21.54,0.00,-10.00,-258.37,-182.07,0.00,0.00,0
-43.56,-8.03,-10.00,10.00,-201.25,-137.51,0.00,0.00,0
36.60,-10.93,-10.00,5.00,-132.36,-211.50,0.00,0.00,0
35.50,14.74,-5.00,-5.00,-169.44,-163.12,0.00,0.00,0
47.09,-26.71,10.00,5.00,-256.78,-214.82,0.00,0.00,0
11.66,37.03,5.00,-10.00,-73.42,23.26,0.00,0.00,0
45.98,-22.00,0.00,-10.00,-259.25,-175.52,0.00,0.00,0
-28.54,2.41,-5.00,10.00,-82.75,-115.46,0.00,0.00,0
28.78,-14.07,-5.00,10.00,-232.30,-194.10,0.00,0.00,0
-32.34,34.59,5.00,0.00,-299.00,-126.68,0.00,0.00,0
-40.44,-27.36,10.00,0.00,-272.16,-145.95,0.00,0.00,0
25.08,-49.77,-5.00,-5.00,-303.75,-7.71,0.00,0.00,0
26.96,38.40,10.00,-5.00,-104.72,-14.97,0.00,0.00,0
48.66,-10.69,10.00,5.00,-337.61,-80.06,0.00,0.00,0
5.05,40.54,10.00,5.00,-152.54,-21.68,0.00,0.00,0
49.59,39.93,10.00,-5.00,-56.92,-28.51,0.00,0.00,0
19.08,-27.75,-5.00,-5.00,-156.61,-23.81,0.00,0.00,0
-46.17,-36.72,10.00,10.00,-2.15,-151.46,0.00,0.00,0
32.66,-30.52,0.00,-5.00,-399.97,-180.07,0.00,0.00,0
23.39,10.86,5.00,-5.00,-280.70,-80.01,0.00,0.00,0
43.65,-18.89,10.00,-10.00,-180.46,-49.79,0.00,0.00,0
-14.66,-41.27,10.00,5.00,-295.47,-78.04,0.00,0.00,0
26.62,45.91,-10.00,-10.00,-382.68,-169.56,0.00,0.00,0
40.42,38.43,0.00,-10.00,-141.94,26.05,0.00,0.00,0
-40.15,-36.09,10.00,-10.00,-316.55,-202.04,0.00,0.00,0
47.65,7.01,10.00,10.00,-301.09,-235.53,0.00,0.00,0
-14.72,28.90,-10.00,-5.00,-355.33,-177.44,0.00,0.00,0
-13.21,31.27,-5.00,-5.00,-187.10,-135.85,0.00,0.00,0
-31.32,-42.97,-10.00,10.00,-66.58,-228.31,0.00,0.00,0
45.66,24.25,10.00,5.00,-79.68,-65.62,0.00,0.00,0
-8.82,-45.67,-5.00,5.00,-346.98,-33.55,0.00,0.00,0
24.59,-36.14,5.00,0.00,-363.33,-171.89,0.00,0.00,0
0.22,-47.11,0.00,-5.00,-242.43,-17.18,0.00,0.00,0
-31.06,-43.13,0.00,10.00,-168.58,8.81,0.00,0.00,0
-48.05,-2.56,0.00,-5.00,-121.51,-236.81,0.00,0.00,0
-47.72,-21.15,5.00,5.00,-156.57,-112.72,0.00,0.00,0
26.79,19.09,10.00,5.00,-99.58,-156.92,0.00,0.00,0
28.56,39.67,0.00,5.00,-359.17,-3.90,0.00,0.00,0
36.39,-47.10,5.00,-10.00,-384.98,38.67,0.00,0.00,0
-5.55,-39.26,0.00,10.00,-212.88,-19.25,0.00,0.00,0
26.12,-38.50,0.00,0.00,-120.73,-227.36,0.00,0.00,0
26.41,29.25,-5.00,-5.00,-57.15,-63.73,0.00,0.00,0
-43.87,0.10,-5.00,-5.00,-187.56,-61.28,0.00,0.00,0
2.84,-30.45,0.00,5.00,-75.21,-119.37,0.00,0.00,0
-35.83,-43.40,10.00,-5.00,-93.08,34.60,0.00,0.00,0
28.78,32.35,-5.00,0.00,-344.47,-126.51,0.00,0.00,0
43.15,-13.48,-10.00,-10.00,-165.13,37.59,0.00,0.00,0
-6.20,25.87,-5.00,10.00,-187.43,-32.26,0.00,0.00,0
-43.89,10.70,5.00,5.00,-246.43,17.10,0.00,0.00,0
-43.74,-25.90,0.00,5.00,-238.06,-56.45,0.00,0.00,0
-40.15,24.86,-10.00,-5.00,-72.43,-102.75,0.00,0.00,0
-18.57,-22.40,10.00,0.00,-314.44,-114.23,0.00,0.00,0
4.13,29.66,0.00,-10.00,-87.21,8.69,0.00,0.00,0
36.94,-36.92,-10.00,10.00,-74.76,-114.22,0.00,0.00,0
-33.88,17.26,-10.00,0.00,-263.93,-22.90,0.00,0.00,0
-25.32,-36.82,-10.00,10.00,-44.32,-223.93,0.00,0.00,0
43.86,3.64,10.00,5.00,-50.53,-135.72,0.00,0.00,0
-32.09,-14.47,5.00,0.00,-49.25,-34.87,0.00,0.00,0
-16.18,23.59,-5.00,5.00,-307.71,-90.06,0.00,0.00,0
-37.28,41.66,0.00,0.00,-167.29,-60.28,0.00,0.00,0
35.76,25.30,0.00,10.00,-145.77,-241.62,0.00,0.00,0
44.13,-44.70,5.00,0.00,-297.09,-0.64,0.00,0.00,0
42.33,30.83,0.00,-5.00,-349.31,-58.39,0.00,0.00,0
-31.43,-35.98,5.00,0.00,-62.46,-146.69,0.00,0.00,0
-11.51,-27.47,5.00,5.00,-156.35,-15.46,0.00,0.00,0
-44.65,-12.51,0.00,5.00,-284.74,16.89,0.00,0.00,0
49.79,10.63,5.00,5.00,-342.10,3.62,0.00,0.00,0
35.28,42.04,0.00,10.00,-65.99,35.15,0.00,0.00,0
-48.12,42.33,-5.00,5.00,-357.21,-75.60,0.00,0.00,0
-35.32,-20.26,5.00,-10.00,-187.68,-24.43,0.00,0.00,0
-48.43,-41.35,10.00,-5.00,-77.43,-41.81,0.00,0.00,0
-24.17,35.77,-5.00,10.00,-377.17,-94.31,0.00,0.00,0
48.34,9.40,10.00,5.00,-163.05,-100.72,0.00,0.00,0
43.24,-41.07,-10.00,5.00,-293.80,-122.61,0.00,0.00,0
-0.09,-48.48,10.00,0.00,-138.08,-42.65,0.00,0.00,0
42.56,37.72,-5.00,-10.00,-143.92,31.69,0.00,0.00,0
23.44,-17.43,10.00,10.00,-127.15,-127.09,0.00,0.00,0
-49.15,8.09,10.00,-5.00,-363.69,-166.25,0.00,0.00,0
-11.09,-33.97,0.00,10.00,-139.11,-192.88,0.00,0.00,0
-42.87,-4.43,10.00,0.00,-361.23,-35.87,0.00,0.00,0
-9.18,-46.26,0.00,0.00,-223.82,42.80,0.00,0.00,0
34.15,-42.81,-10.00,0.00,-171.14,-84.93,0.00,0.00,0
35.01,-7.46,5.00,10.00,-257.68,-191.70,0.00,0.00,0
-35.06,21.28,0.00,5.00,-308.39,-224.58,0.00,0.00,0
-0.34,48.64,10.00,10.00,-108.73,-48.21,0.00,0.00,0
48.32,11.53,-5.00,0.00,-27.37,-111.40,0.00,0.00,0
24.72,39.07,-10.00,-5.00,-158.73,-43.35,0.00,0.00,0
20.34,31.54,-10.00,-5.00,-148.93,48.13,0.00,0.00,0
-47.60,34.41,10.00,-10.00,-279.35,-245.05,0.00,0.00,0
18.52,38.75,5.00,-5.00,-181.72,-71.67,0.00,0.00,0
36.77,-19.86,-10.00,5.00,-86.52,35.15,0.00,0.00,0
-37.07,-38.34,-10.00,0.00,-392.11,37.82,0.00,0.00,0
-23.04,20.71,0.00,-10.00,-7.79,-133.01,0.00,0.00,0
-24.12,10.07,10.00,10.00,-132.84,-153.63,0.00,0.00,0
29.36,28.80,5.00,-5.00,-20.88,-101.65,0.00,0.00,0
-18.67,29.14,10.00,10.00,-279.45,-82.65,0.00,0.00,0
36.62,9.89,5.00,5.00,-118.71,-132.15,0.00,0.00,0
-34.67,-45.58,0.00,-5.00,-289.60,-213.49,0.00,0.00,0
-39.84,-39.11,5.00,10.00,-129.79,-98.23,0.00,0.00,0
-29.38,29.66,10.00,0.00,-213.72,-29.83,0.00,0.00,0
-25.77,12.01,-5.00,-10.00,-205.31,-103.42,0.00,0.00,0
32.14,42.51,-10.00,5.00,-357.04,-34.46,0.00,0.00,0
-43.26,-34.30,-10.00,0.00,-19.18,-184.75,0.00,0.00,0
7.30,-48.12,0.00,5.00,-271.48,-129.53,0.00,0.00,0
-41.31,48.31,0.00,0.00,-202.83,43.58,0.00,0.00,0
-28.67,24.67,5.00,-5.00,-394.94,-29.95,0.00,0.00,0
-25.70,-18.40,5.00,0.00,-181.24,-187.65,0.00,0.00,0
49.42,46.94,0.00,0.00,-92.43,-5.14,0.00,0.00,0
-37.57,-26.88,5.00,-5.00,-344.77,-165.73,0.00,0.00,0
-42.58,39.30,0.00,10.00,-259.58,-57.87,0.00,0.00,0
38.95,-13.60,-10.00,-5.00,-382.25,-72.70,0.00,0.00,0
26.95,-9.82,10.00,-10.00,-290.18,-125.76,0.00,0.00,0
-28.89,-7.36,5.00,-10.00,-361.55,-186.88,0.00,0.00,0
-32.77,45.38,10.00,0.00,-15.46,-54.52,0.00,0.00,0
-38.26,-17.00,0.00,-10.00,-314.26,-12.78,0.00,0.00,0
-12.16,-22.12,-5.00,5.00,-356.45,29.40,0.00,0.00,0
34.44,44.33,10.00,-5.00,-262.21,-180.21,0.00,0.00,0
28.68,-24.50,-10.00,0.00,-246.73,-29.20,0.00,0.00,0
-38.06,41.52,-10.00,-10.00,-63.14,-230.13,0.00,0.00,0
13.51,-39.94,-10.00,10.00,-311.53,-119.29,0.00,0.00,0
-11.34,47.47,-10.00,-10.00,-340.28,-97.64,0.00,0.00,0
16.71,-41.55,10.00,-10.00,-296.44,-245.86,0.00,0.00,0
-26.20,-38.33,0.00,10.00,-198.17,-171.84,0.00,0.00,0
-45.78,-18.75,10.00,-5.00,-200.28,-210.14,0.00,0.00,0
13.13,27.28,10.00,-10.00,-331.99,-88.54,0.00,0.00,0
-43.63,44.97,-10.00,10.00,-157.94,-209.42,0.00,0.00,0
6.46,41.31,10.00,-10.00,-375.13,-88.09,0.00,0.00,0
31.51,-8.61,-5.00,-5.00,-371.31,-191.20,0.00,0.00,0
-25.27,35.14,-10.00,0.00,-200.91,-189.61,0.00,0.00,0
-45.43,-7.61,-5.00,0.00,-248.21,-143.62,0.00,0.00,0
-26.41,-12.19,0.00,-5.00,-399.79,16.27,0.00,0.00,0
7.36,-43.54,0.00,-10.00,-58.55,-220.32,0.00,0.00,0
-34.10,20.47,10.00,5.00,-202.76,-176.21,0.00,0.00,0
-37.56,46.51,5.00,10.00,-137.91,-31.41,0.00,0.00,0
32.20,-11.12,0.00,-10.00,-276.84,-176.48,0.00,0.00,0
-45.30,32.16,-10.00,0.00,-288.32,-39.29,0.00,0.00,0
40.89,-11.05,10.00,-10.00,-147.36,-215.50,0.00,0.00,0
-9.02,-29.59,0.00,0.00,-193.01,-166.96,0.00,0.00,0
42.31,-12.05,10.00,0.00,-280.97,-115.86,0.00,0.00,0
-12.06,49.37,0.00,-5.00,-340.79,-174.27,0.00,0.00,0
-48.58,5.08,5.00,10.00,-258.36,-134.30,0.00,0.00,0
30.80,37.33,-10.00,-5.00,-140.40,-2.99,0.00,0.00,0
32.71,-38.95,-10.00,-10.00,-356.19,-164.49,0.00,0.00,0
-35.52,29.56,0.00,-5.00,-24.58,-181.71,0.00,0.00,0
26.79,-44.87,-5.00,5.00,-142.94,-151.52,0.00,0.00,0
45.74,44.36,5.00,10.00,-352.10,-223.30,0.00,0.00,0
41.15,-15.55,10.00,10.00,-374.83,-65.56,0.00,0.00,0
-37.44,-28.04,-10.00,-5.00,-367.34,-109.52,0.00,0.00,0
39.62,34.27,0.00,-5.00,-372.49,-25.70,0.00,0.00,0
-47.22,-32.74,-5.00,0.00,-14.85,-174.18,0.00,0.00,0
32.41,40.25,5.00,10.00,-289.07,11.91,0.00,0.00,0
-11.40,45.30,-10.00,-5.00,-122.63,-10.91,0.00,0.00,0
-18.59,-33.49,-5.00,-5.00,-6.80,24.06,0.00,0.00,0
4.55,-26.00,-10.00,-10.00,-46.66,6.32,0.00,0.00,0
47.36,-10.81,5.00,-5.00,-370.46,-178.20,0.00,0.00,0
-29.63,35.49,10.00,10.00,-26.29,-11.47,0.00,0.00,0
-42.57,21.76,10.00,10.00,-388.86,-121.05,0.00,0.00,0
32.08,42.11,-10.00,0.00,-289.73,41.98,0.00,0.00,0
-29.34,-6.01,-5.00,0.00,-120.94,-151.02,0.00,0.00,0
-28.55,18.28,10.00,10.00,-374.61,23.70,0.00,0.00,0
15.75,-21.27,-10.00,10.00,-294.87,49.22,0.00,0.00,0
13.00,28.56,0.00,5.00,-140.83,-229.54,0.00,0.00,0
-33.63,-5.23,10.00,-5.00,-224.42,-185.01,0.00,0.00,0
24.55,-48.29,10.00,0.00,-207.72,-8.38,0.00,0.00,0
-21.72,-27.81,10.00,10.00,-363.05,-180.06,0.00,0.00,0
7.62,43.03,0.00,10.00,-32.75,24.21,0.00,0.00,0
29.22,7.08,-10.00,0.00,-157.74,10.45,0.00,0.00,0
-41.61,30.35,-10.00,10.00,-276.54,32.37,0.00,0.00,0
-42.30,29.13,10.00,-10.00,-188.73,-33.75,0.00,0.00,0
12.95,-26.69,-5.00,10.00,-286.90,-121.22,0.00,0.00,0
32.25,12.60,-5.00,0.00,-242.51,-71.32,0.00,0.00,0
-10.16,26.91,0.00,5.00,-10.21,-176.53,0.00,0.00,0
28.42,35.70,0.00,0.00,-80.39,-133.41,0.00,0.00,0
-26.73,-12.64,10.00,10.00,-319.86,-85.92,0.00,0.00,0
-5.21,28.35,10.00,0.00,-9.16,-22.54,0.00,0.00,0
-22.00,33.53,10.00,0.00,-80.43,-158.58,0.00,0.00,0
-36.69,33.77,-5.00,-10.00,-366.69,-158.18,0.00,0.00,0
25.25,11.97,-5.00,5.00,-227.77,3.99,0.00,0.00,0
43.67,-24.64,-5.00,10.00,-131.97,-20.35,0.00,0.00,0
-41.92,-37.87,-10.00,-5.00,-98.49,37.39,0.00,0.00,0
9.38,-44.42,-10.00,10.00,-200.08,-78.00,0.00,0.00,0
-47.27,-19.57,-10.00,5.00,-226.65,-228.22,0.00,0.00,0
34.36,2.09,0.00,0.00,-137.54,20.33,0.00,0.00,0
-33.92,8.12,10.00,5.00,-175.83,-73.18,0.00,0.00,0
-43.05,33.68,-5.00,10.00,-359.64,-53.78,0.00,0.00,0
-8.95,-46.02,0.00,5.00,-100.41,46.93,0.00,0.00,0
22.44,36.89,-5.00,5.00,-107.34,-136.30,0.00,0.00,0
-25.28,-10.12,5.00,0.00,-142.16,-11.30,0.00,0.00,0
18.82,-39.30,-10.00,-10.00,-147.72,-103.02,0.00,0.00,0
42.38,45.06,-5.00,-5.00,-114.12,-182.86,0.00,0.00,0
-21.21,-48.08,0.00,5.00,-399.21,10.66,0.00,0.00,0
-46.71,24.07,5.00,5.00,-345.77,32.58,0.00,0.00,0
47.36,-27.09,10.00,-10.00,-320.28,-19.29,0.00,0.00,0
41.77,-42.35,10.00,0.00,-51.09,-216.37,0.00,0.00,0
28.54,-42.32,-10.00,10.00,-89.57,-70.62,0.00,0.00,0
2.50,-44.89,10.00,10.00,-278.25,-37.72,0.00,0.00,0
-13.97,-43.68,10.00,-5.00,-79.29,-115.61,0.00,0.00,0
27.30,3.02,5.00,-5.00,-214.87,-222.25,0.00,0.00,0
49.47,-48.96,-5.00,5.00,-308.89,-161.75,0.00,0.00,0
-46.77,-41.81,5.00,5.00,-96.83,-141.74,0.00,0.00,0
-26.93,46.29,-5.00,-5.00,-197.66,-182.27,0.00,0.00,0
-27.24,-15.03,0.00,10.00,-373.38,-153.47,0.00,0.00,0
-37.42,-23.05,-5.00,10.00,-261.90,-45.66,0.00,0.00,0
37.69,-5.25,10.00,10.00,-300.19,-244.71,0.00,0.00,0
48.00,-22.86,5.00,-5.00,-85.87,-132.92,0.00,0.00,0
37.53,19.89,-5.00,5.00,-200.15,-30.08,0.00,0.00,0
44.80,-49.70,-5.00,-10.00,-72.12,-137.65,0.00,0.00,0
27.42,18.79,10.00,-5.00,-289.03,-187.22,0.00,0.00,0
46.35,37.72,-10.00,-10.00,-4.99,-114.35,0.00,0.00,0
-47.82,43.26,5.00,-10.00,-345.09,-212.55,0.00,0.00,0
-40.26,9.48,-10.00,5.00,-101.61,-181.37,0.00,0.00,0
-24.01,-26.41,0.00,10.00,-34.05,-93.22,0.00,0.00,0
-22.33,24.11,0.00,0.00,-389.11,-25.63,0.00,0.00,0
32.01,-36.60,10.00,10.00,-163.14,-128.46,0.00,0.00,0
19.67,-19.15,-10.00,-5.00,-293.32,-78.55,0.00,0.00,0
-22.55,35.25,10.00,-10.00,-293.43,-118.80,0.00,0.00,0
20.99,33.90,10.00,10.00,-141.40,-48.45,0.00,0.00,0
26.05,-2.30,0.00,10.00,-313.51,-114.77,0.00,0.00,0
-32.47,3.16,-10.00,-5.00,-194.30,-169.00,0.00,0.00,0
46.32,-17.46,-5.00,-5.00,-384.84,-169.41,0.00,0.00,0
36.21,-4.78,10.00,-5.00,-265.85,-101.01,0.00,0.00,0
-18.97,-47.18,0.00,-10.00,-268.99,-116.24,0.00,0.00,0
-36.24,6.71,-10.00,10.00,-38.92,-124.24,0.00,0.00,0
-23.45,14.36,-10.00,0.00,-307.27,-114.68,0.00,0.00,0
6.79,43.79,10.00,10.00,-30.82,-80.59,0.00,0.00,0
21.16,17.54,0.00,-5.00,-7.42,-10.38,0.00,0.00,0
-37.87,18.16,5.00,5.00,-16.34,-29.36,0.00,0.00,0
34.22,-32.39,-5.00,-10.00,-246.18,-203.14,0.00,0.00,0
29.89,31.24,10.00,-5.00,-141.25,-25.82,0.00,0.00,0
-40.73,-16.54,10.00,10.00,-89.05,-27.25,0.00,0.00,0
9.66,38.10,10.00,10.00,-100.57,-59.87,0.00,0.00,0
-37.87,-10.76,5.00,10.00,-390.07,-52.47,0.00,0.00,0
-32.72,22.84,-10.00,0.00,-191.23,-216.58,0.00,0.00,0
-39.99,36.54,-10.00,0.00,-277.39,4.03,0.00,0.00,0
36.01,22.85,5.00,5.00,-347.37,-72.83,0.00,0.00,0
47.96,10.93,-5.00,-10.00,-259.48,-152.36,0.00,0.00,0
43.81,-17.90,5.00,0.00,-221.86,-241.04,0.00,0.00,0
44.09,15.45,-5.00,-5.00,-55.90,-50.25,0.00,0.00,0
16.59,-38.06,-5.00,10.00,-327.27,-41.42,0.00,0.00,0
11.23,48.96,5.00,-5.00,-15.30,19.73,0.00,0.00,0
0.65,41.09,5.00,-5.00,-4.20,-217.95,0.00,0.00,0
20.16,-38.84,0.00,0.00,-58.65,-169.00,0.00,0.00,0
30.11,26.28,5.00,-5.00,-64.96,-153.62,0.00,0.00,0
-39.19,-45.64,-5.00,-5.00,-376.06,-225.10,0.00,0.00,0
-22.52,-38.68,-5.00,-5.00,-71.76,-194.29,0.00,0.00,0
45.53,-34.11,-10.00,10.00,-68.51,-127.98,0.00,0.00,0
-0.30,-38.08,5.00,0.00,-106.05,-190.10,0.00,0.00,0
41.04,24.25,5.00,5.00,-309.33,-88.73,0.00,0.00,0
-45.41,14.38,-10.00,0.00,-92.33,-95.00,0.00,0.00,0
31.76,14.15,5.00,10.00,-112.53,46.94,0.00,0.00,0
-2.01,49.20,0.00,10.00,-48.03,-142.75,0.00,0.00,0
-19.34,-24.63,-5.00,0.00,-132.48,-209.64,0.00,0.00,0
-46.70,-33.64,5.00,0.00,-89.74,-215.33,0.00,0.00,0
12.12,-41.82,-5.00,10.00,-372.63,42.25,0.00,0.00,0
32.23,-29.48,-10.00,-10.00,-99.11,-206.21,0.00,0.00,0
-11.04,39.78,5.00,0.00,-15.03,-34.58,0.00,0.00,0
30.98,17.39,0.00,-5.00,-163.96,-205.65,0.00,0.00,0
35.88,10.18,10.00,-10.00,-324.63,-223.49,0.00,0.00,0
17.91,-35.06,-10.00,0.00,-241.51,-17.97,0.00,0.00,0
44.45,34.45,0.00,-5.00,-50.07,-39.21,0.00,0.00,0
32.13,4.90,-5.00,5.00,-183.44,-12.67,0.00,0.00,0
-43.30,-16.74,10.00,5.00,-316.86,-140.44,0.00,0.00,0
-38.36,-47.25,5.00,0.00,-6.56,46.84,0.00,0.00,0
44.45,-1.32,5.00,0.00,-286.66,-149.26,0.00,0.00,0
37.48,-45.66,-5.00,10.00,-72.00,-194.05,0.00,0.00,0
-8.06,-35.34,0.00,-10.00,-207.51,-46.60,0.00,0.00,0
18.15,20.48,10.00,-10.00,-227.89,-229.21,0.00,0.00,0
32.28,-20.91,-10.00,-5.00,-120.70,-88.90,0.00,0.00,0
-26.43,-0.24,-5.00,10.00,-104.50,-118.15,0.00,0.00,0
-34.91,-32.40,-10.00,-5.00,-238.94,24.04,0.00,0.00,0
45.10,-18.33,10.00,10.00,-277.24,-34.10,0.00,0.00,0
-27.08,0.67,0.00,5.00,-201.24,-26.18,0.00,0.00,0
-38.51,19.13,-10.00,-10.00,-8.37,-49.29,0.00,0.00,0
13.65,36.59,5.00,-5.00,-255.33,-192.63,0.00,0.00,0
38.04,29.48,10.00,-5.00,-123.74,-146.90,0.00,0.00,0
44.29,-40.30,0.00,-10.00,-133.15,-80.87,0.00,0.00,0
-47.86,-17.03,5.00,10.00,-157.41,-173.56,0.00,0.00,0
39.77,-36.48,5.00,10.00,-53.75,34.54,0.00,0.00,0
39.58,-46.99,-5.00,5.00,-11.31,-29.35,0.00,0.00,0
24.45,-6.21,5.00,0.00,-226.55,32.26,0.00,0.00,0
-23.59,-19.48,5.00,0.00,-223.51,34.20,0.00,0.00,0
41.67,-30.01,-5.00,5.00,-209.69,-44.85,0.00,0.00,0
-36.93,3.58,5.00,-5.00,-29.34,-43.16,0.00,0.00,0
2.58,-30.75,-10.00,10.00,-63.09,-214.59,0.00,0.00,0
4.17,44.38,0.00,5.00,-232.35,-237.23,0.00,0.00,0
22.43,45.65,0.00,-10.00,-371.62,-32.43,0.00,0.00,0
11.88,-44.58,-10.00,5.00,-70.97,-244.49,0.00,0.00,0
35.54,49.23,0.00,5.00,-308.34,-132.36,0.00,0.00,0
-38.39,3.98,10.00,-10.00,-335.57,-68.23,0.00,0.00,0
26.70,0.79,0.00,-5.00,-330.67,-11.64,0.00,0.00,0
-35.32,-27.47,0.00,5.00,-98.92,-20.91,0.00,0.00,0
-34.60,-25.44,10.00,-5.00,-201.02,-240.47,0.00,0.00,0
43.42,20.29,5.00,10.00,-345.43,-56.68,0.00,0.00,0
43.20,40.68,-5.00,-5.00,-19.12,-132.38,0.00,0.00,0
-49.62,-28.93,5.00,-5.00,-125.43,38.39,0.00,0.00,0
-34.90,41.30,-10.00,5.00,-126.35,-44.79,0.00,0.00,0
-17.80,-24.84,-5.00,0.00,-363.14,26.19,0.00,0.00,0
46.79,-24.87,-10.00,-5.00,-56.28,-159.68,0.00,0.00,0
25.98,34.78,10.00,-5.00,-5.61,-7.09,0.00,0.00,0
-42.42,-29.46,-5.00,0.00,-150.05,-42.22,0.00,0.00,0
23.94,20.14,-5.00,10.00,-53.93,-209.18,0.00,0.00,0
35.75,42.61,-5.00,5.00,-214.35,-150.48,0.00,0.00,0
29.59,15.93,0.00,-5.00,-13.81,-198.20,0.00,0.00,0
-19.70,32.84,0.00,10.00,-108.93,-6.03,0.00,0.00,0
-40.04,12.99,10.00,0.00,-234.28,-202.44,0.00,0.00,0
-30.51,-3.79,5.00,-10.00,-114.19,-98.06,0.00,0.00,0
32.08,20.21,10.00,5.00,-359.96,-135.78,0.00,0.00,0
-36.77,-14.44,-5.00,10.00,-5.55,-121.75,0.00,0.00,0
40.10,45.20,5.00,10.00,-105.61,-129.99,0.00,0.00,0
-43.54,-14.26,5.00,-10.00,-251.40,-153.39,0.00,0.00,0
4.78,-41.36,5.00,10.00,-292.41,-12.11,0.00,0.00,0
23.83,24.25,0.00,0.00,-369.19,-19.51,0.00,0.00,0
9.67,-49.67,-5.00,-5.00,-340.57,-92.02,0.00,0.00,0
-18.75,43.66,5.00,10.00,-306.12,-18.44,0.00,0.00,0
-17.79,-48.26,10.00,5.00,-247.65,-90.71,0.00,0.00,0
-32.26,48.89,-5.00,10.00,-163.95,-123.29,0.00,0.00,0
37.62,-9.05,10.00,5.00,-393.46,-215.39,0.00,0.00,0
-5.05,38.38,5.00,-5.00,-165.64,-84.60,0.00,0.00,0
-39.25,4.16,-5.00,-10.00,-45.12,-202.89,0.00,0.00,0
-47.16,-43.87,0.00,5.00,-387.84,-240.28,0.00,0.00,0
28.35,-23.06,5.00,5.00,-370.13,-44.62,0.00,0.00,0
-35.80,43.40,-10.00,5.00,-91.40,0.13,0.00,0.00,0
-21.26,-47.46,-5.00,5.00,-55.78,-13.09,0.00,0.00,0
40.68,-1.78,-5.00,-10.00,-328.96,-194.94,0.00,0.00,0
-37.19,17.17,0.00,-10.00,-270.20,-168.48,0.00,0.00,0
-45.67,41.26,-10.00,0.00,-397.88,-48.48,0.00,0.00,0
48.09,11.15,0.00,-5.00,-196.50,14.97,0.00,0.00,0
40.52,49.89,5.00,5.00,-14.51,-163.88,0.00,0.00,0
49.79,4.74,-10.00,10.00,-131.02,-87.48,0.00,0.00,0
-30.46,-48.09,-5.00,10.00,-14.92,-55.06,0.00,0.00,0
17.06,22.54,-5.00,-10.00,-19.56,27.75,0.00,0.00,0
-45.94,-21.00,5.00,5.00,-333.30,-99.51,0.00,0.00,0
-8.47,-25.98,-5.00,-10.00,-199.67,26.37,0.00,0.00,0
29.39,-26.42,10.00,5.00,-146.75,33.28,0.00,0.00,0
-26.20,33.24,-5.00,0.00,-333.16,15.57,0.00,0.00,0
-17.61,39.97,10.00,5.00,-399.94,-222.61,0.00,0.00,0
36.33,32.22,0.00,5.00,-193.22,-115.94,0.00,0.00,0
15.62,26.02,0.00,5.00,-212.84,-78.51,0.00,0.00,0
30.01,43.94,-10.00,10.00,-138.44,-80.94,0.00,0.00,0
21.90,-38.88,10.00,5.00,-307.83,-133.57,0.00,0.00,0
-19.70,27.81,-5.00,-10.00,-364.34,-213.19,0.00,0.00,0
21.48,37.17,0.00,5.00,-348.77,36.57,0.00,0.00,0
12.07,-41.43,10.00,5.00,-16.65,-191.75,0.00,0.00,0
-15.87,-20.33,-5.00,-5.00,-9.10,-78.95,0.00,0.00,0
16.38,29.31,10.00,-5.00,-188.47,-34.44,0.00,0.00,0
-15.57,35.93,5.00,0.00,-285.44,9.74,0.00,0.00,0
-32.79,44.06,10.00,10.00,-298.58,9.48,0.00,0.00,0
21.98,-33.62,10.00,-10.00,-393.26,-1.56,0.00,0.00,0
-42.11,-16.81,10.00,0.00,-301.02,-246.49,0.00,0.00,0
-49.36,11.91,-10.00,-5.00,-117.00,19.05,0.00,0.00,0
21.92,-42.54,5.00,0.00,-105.17,-131.21,0.00,0.00,0
17.26,40.33,-10.00,-10.00,-38.34,17.20,0.00,0.00,0
37.95,-17.37,-5.00,-5.00,-101.48,-149.39,0.00,0.00,0
36.38,18.16,0.00,-10.00,-38.98,-208.70,0.00,0.00,0
41.26,36.14,-5.00,5.00,-182.47,-224.61,0.00,0.00,0
5.26,-47.57,10.00,-10.00,-285.93,-182.72,0.00,0.00,0
-37.02,-16.32,-5.00,-5.00,-99.73,6.46,0.00,0.00,0
-25.75,30.56,-10.00,5.00,-344.62,-53.61,0.00,0.00,0
-19.60,-21.73,5.00,0.00,-117.84,-236.22,0.00,0.00,0
33.33,-22.47,0.00,-10.00,-87.30,-243.25,0.00,0.00,0
31.72,42.89,-5.00,0.00,-125.38,-36.99,0.00,0.00,0
-32.43,-15.73,5.00,0.00,-143.70,-70.31,0.00,0.00,0
-45.33,-46.69,5.00,-5.00,-354.09,-71.00,0.00,0.00,0
-48.01,30.36,5.00,10.00,-16.40,-92.42,0.00,0.00,0
49.76,12.61,-5.00,-5.00,-355.44,-248.61,0.00,0.00,0
47.52,-28.70,-5.00,5.00,-79.39,-37.98,0.00,0.00,0
-33.86,-9.05,-10.00,-5.00,-159.52,-242.65,0.00,0.00,0
-4.12,44.86,-10.00,-10.00,-354.21,-87.95,0.00,0.00,0
-22.16,-15.91,5.00,0.00,-370.87,-143.37,0.00,0.00,0
26.58,-4.51,-10.00,5.00,-109.20,-8.56,0.00,0.00,0
-48.87,3.89,10.00,0.00,-5.60,-162.09,0.00,0.00,0
28.22,-10.79,-5.00,-10.00,-347.70,47.37,0.00,0.00,0
-32.11,37.27,5.00,-5.00,-140.92,-93.37,0.00,0.00,0
18.00,-43.73,0.00,5.00,-209.11,20.66,0.00,0.00,0
36.26,12.33,-5.00,-5.00,-253.69,-131.50,0.00,0.00,0
-0.45,38.64,0.00,-10.00,-241.93,39.25,0.00,0.00,0
48.60,3.99,-10.00,-5.00,-165.23,-18.66,0.00,0.00,0
28.32,-11.87,-5.00,10.00,-22.01,-143.28,0.00,0.00,0
1.54,-43.74,5.00,0.00,-26.28,44.33,0.00,0.00,0
33.21,0.07,5.00,-5.00,-109.73,-92.09,0.00,0.00,0
25.74,-11.31,-10.00,10.00,-59.91,-124.39,0.00,0.00,0
-15.72,-48.25,0.00,-5.00,-217.91,-48.99,0.00,0.00,0
-25.62,15.17,10.00,5.00,-280.70,-138.38,0.00,0.00,0
2.51,-40.70,5.00,0.00,-288.06,-14.34,0.00,0.00,0
-40.68,-38.95,-5.00,-5.00,-199.98,-164.15,0.00,0.00,0
40.24,-13.17,10.00,5.00,-206.90,28.84,0.00,0.00,0
32.47,-39.37,10.00,10.00,-342.79,-159.17,0.00,0.00,0
-29.38,-13.20,5.00,-5.00,-227.68,-213.79,0.00,0.00,0
29.16,-0.06,5.00,-10.00,-80.03,-182.36,0.00,0.00,0
-11.41,-30.50,-5.00,5.00,-230.05,48.90,0.00,0.00,0
24.23,-33.23,-5.00,5.00,-232.52,-87.99,0.00,0.00,0
39.75,-7.25,5.00,10.00,-273.09,-28.53,0.00,0.00,0
11.54,-24.13,-10.00,5.00,-151.24,-234.71,0.00,0.00,0
-41.59,18.39,10.00,0.00,-315.23,-95.24,0.00,0.00,0
-6.28,-27.65,-5.00,0.00,-330.36,-132.67,0.00,0.00,0
25.53,-14.52,-5.00,0.00,-160.34,-110.63,0.00,0.00,0
-38.26,43.73,-10.00,0.00,-188.54,-113.35,0.00,0.00,0
-37.64,30.31,-5.00,0.00,-167.04,6.26,0.00,0.00,0
42.74,-44.62,5.00,10.00,-66.64,-167.58,0.00,0.00,0
33.49,4.93,10.00,-5.00,-62.19,-41.35,0.00,0.00,0
22.73,14.25,0.00,0.00,-345.33,47.05,0.00,0.00,0
-17.86,31.47,-10.00,-5.00,-57.24,-33.82,0.00,0.00,0
-47.90,19.82,5.00,5.00,-14.25,-74.32,0.00,0.00,0
31.38,43.90,5.00,-10.00,-75.17,-161.98,0.00,0.00,0
26.82,2.38,-5.00,-10.00,-367.13,-162.53,0.00,0.00,0
-38.37,-4.30,-5.00,-10.00,-307.77,-148.69,0.00,0.00,0
-21.70,37.35,-10.00,0.00,-334.27,-162.89,0.00,0.00,0
-28.08,45.30,5.00,0.00,-279.02,-67.38,0.00,0.00,0
-8.64,-38.77,10.00,5.00,-42.22,-45.40,0.00,0.00,0
29.71,-32.63,0.00,0.00,-38.21,-95.12,0.00,0.00,0
-32.87,48.46,-5.00,10.00,-210.01,-203.96,0.00,0.00,0
-27.69,-30.38,5.00,-5.00,-241.21,48.84,0.00,0.00,0
-4.21,43.27,-5.00,0.00,-190.27,-175.20,0.00,0.00,0
21.32,40.13,5.00,-5.00,-261.52,-111.04,0.00,0.00,0
-40.53,-25.99,10.00,10.00,-290.25,-1.05,0.00,0.00,0
-41.57,38.35,0.00,5.00,-7.03,-173.37,0.00,0.00,0
-1.10,-26.56,0.00,5.00,-252.52,-12.79,0.00,0.00,0
-41.66,43.94,-10.00,0.00,-259.88,-150.57,0.00,0.00,0
35.71,13.23,-10.00,-10.00,-299.21,-193.18,0.00,0.00,0
33.44,-15.86,10.00,0.00,-264.67,28.78,0.00,0.00,0
-34.54,48.37,5.00,-10.00,-236.19,-115.45,0.00,0.00,0
-46.43,15.87,-10.00,0.00,-132.70,-187.34,0.00,0.00,0
-27.84,-4.20,10.00,0.00,-361.34,-38.59,0.00,0.00,0
31.03,-28.92,-10.00,5.00,-2.78,38.28,0.00,0.00,0
-11.95,-31.36,5.00,0.00,-351.71,0.18,0.00,0.00,0
8.22,33.37,-5.00,5.00,-286.14,-63.44,0.00,0.00,0
-6.61,-28.37,0.00,5.00,-202.68,-170.60,0.00,0.00,0
-21.44,-14.90,10.00,-10.00,-28.36,-225.51,0.00,0.00,0
-46.16,35.40,-10.00,0.00,-155.19,8.64,0.00,0.00,0
-11.70,47.39,-5.00,0.00,-381.04,-113.21,0.00,0.00,0
43.88,-20.02,-10.00,10.00,-20.72,28.70,0.00,0.00,0
-5.66,34.69,10.00,10.00,-1.33,-118.93,0.00,0.00,0
25.55,-6.61,10.00,-10.00,-382.23,-91.78,0.00,0.00,0
7.69,-33.13,5.00,5.00,-43.14,-163.00,0.00,0.00,0
23.00,36.52,-5.00,0.00,-160.14,-247.83,0.00,0.00,0
47.33,30.22,10.00,5.00,-141.37,-59.07,0.00,0.00,0
25.22,-48.86,0.00,0.00,-113.05,-0.52,0.00,0.00,0
-19.25,20.34,10.00,5.00,-261.03,26.41,0.00,0.00,0
41.90,-11.79,5.00,0.00,-135.92,-229.68,0.00,0.00,0
-3.75,-42.24,5.00,0.00,-83.00,-59.86,0.00,0.00,0
40.63,2.34,0.00,-10.00,-238.48,2.05,0.00,0.00,0
49.57,-22.76,-5.00,-5.00,-165.11,-19.86,0.00,0.00,0
-9.29,42.16,0.00,-10.00,-64.57,-197.49,0.00,0.00,0
-22.56,-15.26,0.00,10.00,-269.60,30.34,0.00,0.00,0
-49.27,10.20,5.00,-5.00,-308.22,-70.12,0.00,0.00,0
49.20,-5.88,-5.00,5.00,-149.10,-169.57,0.00,0.00,0
-11.90,-29.19,-10.00,-5.00,-328.58,-236.63,0.00,0.00,0
34.34,22.77,10.00,-10.00,-138.93,-51.31,0.00,0.00,0
47.92,38.63,-10.00,0.00,-55.56,-247.75,0.00,0.00,0
46.15,-22.79,-10.00,10.00,-164.10,-89.78,0.00,0.00,0
-28.72,-46.44,10.00,10.00,-176.62,10.53,0.00,0.00,0
-37.53,28.54,10.00,-10.00,-23.81,-5.04,0.00,0.00,0
-42.53,44.38,5.00,-10.00,-101.85,-14.30,0.00,0.00,0
-30.31,6.68,10.00,5.00,-48.71,-120.65,0.00,0.00,0
8.10,-42.58,0.00,5.00,-2.24,-180.75,0.00,0.00,0
26.80,15.81,-10.00,10.00,-73.33,-234.79,0.00,0.00,0
30.27,-40.93,10.00,-10.00,-336.77,0.39,0.00,0.00,0
17.30,30.54,10.00,10.00,-141.48,-139.73,0.00,0.00,0
20.15,20.12,5.00,5.00,-177.74,-211.54,0.00,0.00,0
10.21,45.83,10.00,10.00,-88.34,-47.55,0.00,0.00,0
47.87,4.93,10.00,5.00,-295.36,27.17,0.00,0.00,0
-17.01,-45.14,-10.00,0.00,-253.70,-72.81,0.00,0.00,0
27.73,23.94,5.00,-10.00,-133.07,-247.75,0.00,0.00,0
42.82,-42.56,10.00,-10.00,-144.52,-235.83,0.00,0.00,0
38.32,-19.72,-10.00,0.00,-42.48,-79.05,0.00,0.00,0
-49.01,28.91,5.00,0.00,-65.39,-160.00,0.00,0.00,0
18.19,-49.59,-5.00,10.00,-232.89,24.70,0.00,0.00,0
-44.71,-46.32,10.00,0.00,-84.51,-215.17,0.00,0.00,0
-46.46,3.09,5.00,-10.00,-41.78,-127.62,0.00,0.00,0
26.19,36.37,5.00,-5.00,-195.60,-175.75,0.00,0.00,0
-17.40,40.75,5.00,0.00,-67.86,27.26,0.00,0.00,0
11.60,25.47,-5.00,5.00,-386.92,28.73,0.00,0.00,0
-36.32,47.10,5.00,-5.00,-327.59,-198.42,0.00,0.00,0
-28.67,30.87,-5.00,0.00,-179.46,49.39,0.00,0.00,0
20.27,-42.16,10.00,-5.00,-394.72,-220.82,0.00,0.00,0
24.11,-30.09,-10.00,10.00,-198.70,-100.21,0.00,0.00,0
-6.78,-39.61,0.00,0.00,-368.75,24.04,0.00,0.00,0
-36.15,22.37,-10.00,-5.00,-244.70,20.06,0.00,0.00,0
-32.24,-24.89,-10.00,5.00,-297.72,-95.81,0.00,0.00,0
-13.32,22.53,-5.00,0.00,-294.87,-238.62,0.00,0.00,0
26.92,-1.61,-5.00,-5.00,-235.16,-228.43,0.00,0.00,0
33.48,-4.46,10.00,10.00,-130.65,-247.16,0.00,0.00,0
-25.37,-48.44,-5.00,-10.00,-316.28,-78.72,0.00,0.00,0
-49.58,-22.31,5.00,0.00,-143.76,-22.08,0.00,0.00,0
41.10,-21.33,10.00,10.00,-377.39,-53.05,0.00,0.00,0
-20.36,-46.08,0.00,-10.00,-53.20,-181.25,0.00,0.00,0
38.06,-46.17,5.00,-10.00,-10.36,-243.36,0.00,0.00,0
-39.38,20.32,5.00,5.00,-365.41,-187.13,0.00,0.00,0
-33.04,-29.98,-10.00,-5.00,-0.59,38.60,0.00,0.00,0
-32.95,-28.08,-5.00,-5.00,-246.03,-136.01,0.00,0.00,0
26.97,49.12,5.00,5.00,-217.95,-5.69,0.00,0.00,0
47.05,-38.10,-10.00,0.00,-55.50,-155.18,0.00,0.00,0
-10.17,41.43,0.00,-5.00,-204.09,-37.95,0.00,0.00,0
-30.05,-11.37,-5.00,5.00,-134.93,-126.24,0.00,0.00,0
-0.30,-41.90,-10.00,-5.00,-151.49,-196.27,0.00,0.00,0
49.97,-21.50,0.00,5.00,-361.50,-138.47,0.00,0.00,0
29.57,24.59,0.00,5.00,-142.03,-66.80,0.00,0.00,0
-47.94,21.13,0.00,0.00,-165.08,-93.66,0.00,0.00,0
-40.91,-8.95,5.00,-10.00,-290.25,-164.46,0.00,0.00,0
"""
df = pd.read_csv(io.StringIO(csv_data))
print(f'Cargadas {len(df)} muestras')

In [ ]:
# 2. Definir Red Estándar CPT
class TabularNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

model = TabularNet()
opt = optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.BCELoss()

In [ ]:
# 3. Entrenamiento
X = torch.tensor(df.drop('success', axis=1).values, dtype=torch.float32)
y = torch.tensor(df['success'].values, dtype=torch.float32).view(-1, 1)

for epoch in range(200):
    opt.zero_grad()
    outputs = model(X)
    loss = loss_fn(outputs, y)
    loss.backward()
    opt.step()
    if epoch % 50 == 0: print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

torch.save(model.state_dict(), 'newton_tabular_filter.pt')
print('✅ Modelo Newtoniano Guardado')

In [ ]:
# 4. Descargar / Guardar
import shutil, os
output_file = 'newton_tabular_filter.pt'
if os.path.exists('/kaggle/working'):
    print('Model deployed.')
    print('✅ Modelo guardado en Kaggle Output')
else:
    try:
        from google.colab import files
        files.download(output_file)
        print('✅ Modelo descargado desde Colab')
    except ImportError:
        print('✅ Modelo guardado localmente')
